# Export game data
Exports the measurement results that the game page (`docs/index.html`) replays.

- **Real hardware:** downloads the 10,000 results of my ibm_fez job `dbdkskk24mrc739ihmj0`, in the order they were measured. It only reads the stored results, so it uses no new IBM Quantum time.
- **Ideal simulator** and **naive mode:** 10,000 shots each from Qiskit's `StatevectorSampler`.
- Everything is saved to `quantum_results.json` (one digit per shot: 0 = `00` Rock, 1 = `01` Paper, 2 = `10` Scissors, 3 = `11`).

The hardware results match section 7b of `quantum_rps.ipynb`: Rock 3,426 · Paper 3,292 · Scissors 3,219 · `11`: 63.
Run `ibm_setup.ipynb` once first so this notebook can connect to IBM Quantum.

In [1]:
import json
from datetime import datetime, timezone
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit_ibm_runtime import QiskitRuntimeService

SHOTS = 10000
code = {'00': '0', '01': '1', '10': '2', '11': '3'}   # 0 = Rock, 1 = Paper, 2 = Scissors, 3 = the "impossible" 11

# 1. Real hardware: the 10,000 results from my ibm_fez job, in the order they were measured (no new QPU time used)
job = QiskitRuntimeService().job("dbdkskk24mrc739ihmj0")
hw_bits = job.result()[0].data.meas.get_bitstrings()

# 2. Ideal simulator: the same Approach 3 circuit (Ry + controlled-H)
qc3 = QuantumCircuit(2)
qc3.ry(2 * np.arccos(np.sqrt(2/3)), 0)
qc3.ch(0, 1, ctrl_state=0)
qc3.measure_all()
ideal_bits = StatevectorSampler().run([qc3], shots=SHOTS).result()[0].data.meas.get_bitstrings()

# 3. Naive mode: H on both qubits (the game will treat 11 as Rock to show the bias)
qc_naive = QuantumCircuit(2)
qc_naive.h([0, 1])
qc_naive.measure_all()
naive_bits = StatevectorSampler().run([qc_naive], shots=SHOTS).result()[0].data.meas.get_bitstrings()

data = {
    "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "encoding": "one digit per shot, in measurement order: 0=00 Rock, 1=01 Paper, 2=10 Scissors, 3=11",
    "hardware": {"backend": job.backend().name, "job_id": job.job_id(), "shots": len(hw_bits),
                 "results": "".join(code[b] for b in hw_bits)},
    "ideal": {"backend": "Qiskit StatevectorSampler (ideal)", "shots": len(ideal_bits),
              "results": "".join(code[b] for b in ideal_bits)},
    "naive": {"backend": "Qiskit StatevectorSampler (ideal), H on both qubits", "shots": len(naive_bits),
              "results": "".join(code[b] for b in naive_bits)},
}
with open("quantum_results.json", "w") as f:
    json.dump(data, f)

for mode in ("hardware", "ideal", "naive"):
    r = data[mode]["results"]
    print(mode, {name: r.count(d) for d, name in zip("0123", ["Rock", "Paper", "Scissors", "11"])})

qiskit_runtime_service.__init__:WARNING:2026-10-24 10:19:55,122: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


hardware {'Rock': 3426, 'Paper': 3292, 'Scissors': 3219, '11': 63}
ideal {'Rock': 3317, 'Paper': 3395, 'Scissors': 3288, '11': 0}
naive {'Rock': 2526, 'Paper': 2494, 'Scissors': 2483, '11': 2497}
